### Colab Mount


In [ ]:
import os
import sys
from pathlib import Path

try:
    import google.colab
    from google.colab import drive

    IN_COLAB = True
except Exception:
    IN_COLAB = False

if IN_COLAB:
    drive.mount("/content/drive")

cwd = Path.cwd().resolve()
roots = [cwd, *cwd.parents]
src_candidates = []
for root in roots:
    src_candidates.extend(
        [
            root,
            root / "local_learning",
            root / "src" / "local_learning",
            root / "local_learning" / "src",
        ]
    )
src_candidates.extend(
    [
        Path("/content/drive/MyDrive/Research/AI-in-Science-Lab/src/local_learning"),
        Path("/content/drive/MyDrive/Research/AI-In-Science-Lab/src/local_learning"),
        Path("/content/drive/MyDrive/Research/AI-in-Science-Lab/local_learning/src"),
        Path("/content/drive/MyDrive/Research/AI-In-Science-Lab/local_learning/src"),
    ]
)
src_dir = next(
    (path.resolve() for path in src_candidates if (path / "training.py").is_file() and (path / "models").is_dir()),
    None,
)
if src_dir is None:
    raise FileNotFoundError("Could not find local_learning source directory with training.py and models/.")

os.chdir(src_dir)
sys.path = [path for path in sys.path if path != str(src_dir)]
sys.path.insert(0, str(src_dir))
for module_name in list(sys.modules):
    if module_name in {"training", "datasets", "models", "losses", "loggers"}:
        del sys.modules[module_name]
    elif module_name.startswith(("datasets.", "models.", "losses.", "loggers.")):
        del sys.modules[module_name]
print(f"[src] {src_dir}")


### Setup


In [ ]:
import contextlib
import io

import torch
import wandb

from training import train

torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"[device] {DEVICE}")


### Experiment Configs


In [ ]:
VGG16_BASE_CONFIG = {
    "vgg16_first_full_training_epoch": 1,
    "vgg16_deconv_training": False,
    "vgg16_local_training": False,
    "vgg16_small": True,
}

BASE_CONFIG = {
    **VGG16_BASE_CONFIG,
    "project": "local-learning-6",
    "architecture_type": "vgg16",
    "data": "smallcifar10:normalize",
    "loss_type": "regular",
    "learning_rate": 0.006,
    "epochs": 2,
    "batch_size": 64,
    "num_workers": 2,
    "lambda_strength": 0.0,
    "kernel_size": 3,
    "correlation_loss": "sum",
    "local": False,
    "local_reconstruction_strength": 1.0,
    "hidden_channels": [48, 32],
    "num_layers": 2,
    "k_population": None,
    "k_lifetime": 0.2,
    "wta_eval_multiplier": 1.0,
    "log_every_steps": 100,
}


### Sweep


In [ ]:
SWEEP_PROJECT = "local-learning-6"
SWEEP_RUNS = 1
SWEEP_BASE_CONFIG = {**BASE_CONFIG, "project": SWEEP_PROJECT}

SWEEP_VALUES = {}


def wandb_sweep_parameters(base_config: dict, sweep_values: dict) -> dict:
    parameters = {key: {"value": value} for key, value in base_config.items()}
    for key, values in sweep_values.items():
        parameters[key] = {"values": values if isinstance(values, list) else list(values)}
    return parameters


SWEEP_CONFIG = {
    "name": "small-vgg16-smallcifar10-logging-smoke",
    "project": SWEEP_PROJECT,
    "method": "grid",
    "metric": {
        "name": "test/acc",
        "goal": "maximize",
    },
    "parameters": wandb_sweep_parameters(SWEEP_BASE_CONFIG, SWEEP_VALUES),
}


def sweep_train() -> None:
    train(SWEEP_BASE_CONFIG, device=DEVICE)


def run_sweep(*, sweep_id: str | None = None, count: int | None = SWEEP_RUNS) -> str:
    if sweep_id is None:
        with contextlib.redirect_stdout(io.StringIO()), contextlib.redirect_stderr(io.StringIO()):
            sweep_id = wandb.sweep(SWEEP_CONFIG, project=SWEEP_PROJECT)

    with contextlib.redirect_stdout(io.StringIO()), contextlib.redirect_stderr(io.StringIO()):
        wandb.agent(sweep_id, function=sweep_train, count=count)
    return sweep_id


### Run


In [ ]:
sweep_id = run_sweep()
sweep_id
